# Tests: `fasterai.core.precision` (source `nbs/core/precision.ipynb`)

In [ ]:
from fastcore.test import *
from fasterai.core.precision import *
from fasterai.core.precision import _CELLS, _TORCHAO_RECIPES, _label, _resolve_spec, _split_weight_bits

In [ ]:
from fastcore.test import *

# --- the matrix is well formed: every cell is reachable by its own key ---
for key, c in PRECISION_SUPPORT.items():
    test_eq(key, (c.backend, c.weight_bits, c.act_bits))
    assert c.qschemes, f"{key} names no weight axis"
    assert set(c.qschemes) <= set(QSCHEMES), f"{key} names an unknown axis"
    assert set(c.symmetries) <= {True, False}, f"{key} names an unknown symmetry"
    assert c.weight_bits in WIDTHS and c.act_bits in WIDTHS, f"{key} uses a width the grammar cannot name"
    assert c.note.endswith('.'), f"{key} has no explanation"
    # a per-group cell is the only kind allowed to carry a default group size
    if c.default_group_size is not None: assert 'per_group' in c.qschemes, key
    assert set(c.qdq_placements) <= set(QDQ_PLACEMENTS), f"{key} names an unknown Q/DQ placement"
    # a cell that places its pairs at all can place them the ordinary way
    if c.qdq_placements: assert c.default_qdq_placement == 'per_op', key

# every backend but torchao is INT8-only today: a new cell elsewhere needs its kernels checked first
assert all(c.label == 'W8A8' for c in _CELLS if c.backend != 'torchao')

# per-layer widths are a property of the CELL, not of the backend: torchao honors a dict weight-only,
# and never with the run-time activation scales of its W8A8 recipe
test_eq(sorted((c.backend, c.label) for c in _CELLS if c.per_layer),
        [('fbgemm', 'W8A8'), ('onednn', 'W8A8'), ('qnnpack', 'W8A8'), ('torchao', 'W8A16'), ('x86', 'W8A8')])

# exactly one cell exports today, and it is the symmetric pt2e one
test_eq([c.label for c in PRECISION_SUPPORT.values() if c.exports], ['W8A8'])
test_eq([c.backend for c in PRECISION_SUPPORT.values() if c.exports], ['pt2e'])

# choosing where the Q/DQ pairs sit is a property of one CELL, not of the whole grammar: pt2e is the
# only flow fasterai annotates itself, so it is the only one that can move a pair
test_eq(sorted((c.backend, c.label) for c in _CELLS if c.qdq_placements), [('pt2e', 'W8A8')])
test_eq(PRECISION_SUPPORT[('pt2e', 8, 8)].qdq_placements, QDQ_PLACEMENTS)
test_eq(QDQ_PLACEMENTS, ('per_op', 'skip_conv_add'))
for c in _CELLS:
    if not c.qdq_placements: test_eq(c.default_qdq_placement, None)

# defaults come off the front of the tuples
_pt2e = PRECISION_SUPPORT[('pt2e', 8, 8)]
test_eq((_pt2e.default_qscheme, _pt2e.default_symmetric, _pt2e.label), ('per_channel', True, 'W8A8'))
test_eq(_pt2e.default_qdq_placement, 'per_op')
test_eq(_pt2e.as_dict()['backend'], 'pt2e')
test_eq(_pt2e.as_dict()['qdq_placements'], QDQ_PLACEMENTS)
test_eq(_label(4, 16), 'W4A16')

# the rendered table shows every cell, and is generated from the matrix
_table = precision_table()
for c in PRECISION_SUPPORT.values(): assert f"`{c.backend}`" in _table and c.note in _table
test_eq(_table.count('\n'), len(PRECISION_SUPPORT) + 1)  # header + separator + one row per cell
# ...one column per thing a cell can say, header and separator agreeing (8 today: the Q/DQ placement
# column is the newest one, and a cell without that axis renders 'n/a' rather than an empty column)
_head, _sep, *_rows = _table.split('\n')
test_eq(_head.count('|'), 9)                     # 8 columns, 9 pipes
test_eq(_sep.count('|'), _head.count('|'))
for _row in _rows: test_eq(_row.count('|'), _head.count('|'))
assert '| Q/DQ placement |' in _head, _head
assert '| per_op, skip_conv_add |' in _table
assert '| n/a |' in _table

In [ ]:
# --- resolution: the defaults every legacy call relies on ---
test_eq(_resolve_spec('x86').as_dict(),
        {'backend': 'x86', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel',
         'symmetric': False, 'group_size': None, 'layer_bits': None, 'qdq_placement': None})
test_eq(_resolve_spec('pt2e').qscheme, 'per_channel')       # per-channel weights, as pt2e has always done
test_eq(_resolve_spec('pt2e').symmetric, True)
test_eq(_resolve_spec('qnnpack').qscheme, 'per_tensor')     # qnnpack's default observer is per-tensor
test_eq(_resolve_spec('x86', 'qat').method, 'qat')          # `method` is left alone, it is the schedule axis

# a spec asked for explicitly is the same spec as the default one: the grammar is a superset
test_eq(_resolve_spec('pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True),
        _resolve_spec('pt2e'))

# --- Q/DQ placement: an axis only the pt2e cell has ---
test_eq(_resolve_spec('pt2e').qdq_placement, 'per_op')                       # the cell's own default...
test_eq(_resolve_spec('pt2e', qdq_placement='per_op'), _resolve_spec('pt2e'))  # ...spelled out, same spec
test_eq(_resolve_spec('pt2e', qdq_placement='skip_conv_add').qdq_placement, 'skip_conv_add')
assert _resolve_spec('pt2e', qdq_placement='skip_conv_add') != _resolve_spec('pt2e')
# a backend whose flow has no such axis records nothing, the way `group_size` records nothing off
# `per_group` — the field describes the arithmetic, so it may not claim one that never ran
for _backend in ('x86', 'fbgemm', 'onednn', 'qnnpack'):
    test_eq(_resolve_spec(_backend).qdq_placement, None)
test_eq(_resolve_spec('torchao', 'int8_weight_only').qdq_placement, None)

# --- the axes have an effect ---
test_eq(_resolve_spec('pt2e', qscheme='per_tensor').qscheme, 'per_tensor')
test_eq(_resolve_spec('x86', use_per_tensor=True).qscheme, 'per_tensor')  # the legacy flag is an axis request
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=16, qscheme='per_group', group_size=64).group_size, 64)

# --- torchao: the recipe and the precision are two names for one thing ---
test_eq(_resolve_spec('torchao', 'int8_weight_only').label, 'W8A16')
test_eq(_resolve_spec('torchao', 'int8_dynamic').label, 'W8A8')
test_eq(_resolve_spec('torchao', 'int4_weight_only').as_dict()['group_size'], 128)  # its native group size
# ...so naming the precision picks the recipe
test_eq(_resolve_spec('torchao', weight_bits=4).method, 'int4_weight_only')
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=16).method, 'int8_weight_only')
test_eq(_resolve_spec('torchao', weight_bits=8, act_bits=8).method, 'int8_dynamic')
test_eq(_TORCHAO_RECIPES, {'W8A16': 'int8_weight_only', 'W8A8': 'int8_dynamic', 'W4A16': 'int4_weight_only'})

# --- per-layer widths ---
_spec = _resolve_spec('x86', weight_bits={'fc': 16})
test_eq((_spec.weight_bits, _spec.layer_bits), (8, {'fc': 16}))   # unlisted layers keep the backend default
_asked = {'fc': 16}
_spec = _resolve_spec('x86', weight_bits=_asked)
_asked['fc'] = 8
test_eq(_spec.layer_bits, {'fc': 16})  # the spec holds a copy, not the caller's dict

# --- torchao honors a per-layer dict weight-only: the width the dict does NOT name is the uniform one ---
_ao = _resolve_spec('torchao', weight_bits={'0': 16, '2': 8}, act_bits=16)
test_eq((_ao.backend, _ao.method, _ao.label), ('torchao', 'int8_weight_only', 'W8A16'))
test_eq((_ao.weight_bits, _ao.layer_bits), (8, {'0': 16, '2': 8}))
# naming the recipe instead of the precision is the same request
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'0': 16, '2': 8}), _ao)

# the scalar is DERIVED from the dict: an 8 anywhere fixes the width at 8, an all-16 dict leaves it to
# the backend (which is what keeps `weight_bits={'fc': 16}` resolving exactly as it always has)
test_eq(_split_weight_bits({'a': 16, 'b': 8}), (8, {'a': 16, 'b': 8}))
test_eq(_split_weight_bits({'a': 16, 'b': 16}), (None, {'a': 16, 'b': 16}))
test_eq(_split_weight_bits(8), (8, None))
test_eq(_split_weight_bits(None), (None, None))
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'0': 16}).weight_bits, 8)

# --- the spec is frozen, and knows what it can export ---
_spec = _resolve_spec('pt2e')
test_eq((_spec.exports, _spec.cell.backend, _spec.label), (True, 'pt2e', 'W8A8'))
test_eq(_resolve_spec('torchao', 'int8_weight_only').exports, False)
with ExceptionExpected(AttributeError): _spec.weight_bits = 4
with ExceptionExpected(AttributeError): _spec.qscheme = 'per_tensor'
assert _spec.note

# --- `quant_spec` is the one way to read the spec back off a model ---
class _Tagged: pass
_model = _Tagged()
test_eq(quant_spec(_model), None)                  # an unquantized model has no precision to report
setattr(_model, SPEC_ATTR, _spec)
test_eq(quant_spec(_model), _spec)
test_eq(SPEC_ATTR, '_fasterai_quant_spec')         # the attribute name is part of the contract

In [ ]:
# --- every refusal is loud, names the argument, and points at a backend that can ---
def _refused(regex, *args, exc=ValueError, **kwargs):
    "Assert `_resolve_spec` refuses this request with a message matching `regex`"
    with ExceptionExpected(exc, regex=regex): _resolve_spec(*args, **kwargs)

# a precision no backend runs
_refused("No fasterai backend runs W4A8", 'pt2e', weight_bits=4, act_bits=8)
_refused("no recipe for W4A8", 'torchao', weight_bits=4, act_bits=8)
# a precision this backend does not run, but another one does
_refused("that run W8A16", 'pt2e', act_bits=16)
# an axis this backend does not offer
_refused("not 'per_channel'", 'qnnpack', qscheme='per_channel')
_refused("Unknown qscheme 'channel'", 'pt2e', qscheme='channel')
# a symmetry its observers cannot produce — the silent-drop case the FX flow used to hide
_refused("cannot honor symmetric=True", 'x86', symmetric=True)
_refused("cannot honor symmetric=True", 'fbgemm', symmetric=True)
# per-group asks and their backend
_refused("only means something with qscheme='per_group'", 'x86', group_size=64)
_refused("needs a `group_size`", 'torchao', weight_bits=8, act_bits=16, qscheme='per_group')
_refused("that do: ", 'pt2e', qscheme='per_group', group_size=64)
# a Q/DQ placement on a backend whose flow does not place its pairs — including the DEFAULT placement,
# which on such a cell is a request nothing would read rather than a harmless no-op
_refused("names no placement at all", 'x86', qdq_placement='skip_conv_add')
_refused("names no placement at all", 'x86', qdq_placement='per_op')
_refused("The backend\\(s\\) that can: \\['pt2e'\\]", 'torchao', 'int8_weight_only',
         qdq_placement='skip_conv_add')
_refused("Unknown qdq_placement 'fuse_residuals'", 'pt2e', qdq_placement='fuse_residuals')
_refused("`qdq_placement` must be one of ", 'pt2e', exc=TypeError, qdq_placement=1)
# per-layer widths on a backend that quantizes the whole graph at once
_refused("cannot honor a per-layer `weight_bits` dict", 'pt2e', weight_bits={'fc': 16})
# ...while torchao DOES honor one weight-only, so the dict resolves instead of being refused
test_eq(_resolve_spec('torchao', 'int8_weight_only', weight_bits={'fc': 16}).layer_bits, {'fc': 16})
# ...and its other cell, whose activation scales are computed at run time, names the way to the one
# that can rather than sending the caller to another backend
_refused("only at W8A16: add act_bits=16", 'torchao', weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", 'torchao', 'int8_dynamic', weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", 'torchao', weight_bits={'fc': 16}, act_bits=8)
# a per-layer width is 8-or-16 on every backend: there is no per-layer INT4 in this alphabet
_refused("a per-layer width is 8", 'torchao', weight_bits={'fc': 4}, act_bits=16)
# contradictions between two ways of asking for the same thing
_refused("ask for different weight axes", 'x86', use_per_tensor=True, qscheme='per_channel')
_refused("never read", 'pt2e', use_per_tensor=True)
_refused("contradicts", 'torchao', 'int8_weight_only', act_bits=8)
_refused("has no method 'invalid'", 'torchao', 'invalid')

# --- plausible-wrong values get a clear error, never a cryptic one downstream ---
# every type complaint reads the same: "`name` must be ..., got value (type)."
_refused("`weight_bits` must be an int", 'pt2e', exc=TypeError, weight_bits='8')
_refused("`act_bits` must be an int", 'pt2e', exc=TypeError, act_bits=8.0)
_refused("is not a width this grammar names", 'pt2e', act_bits=32)
_refused("is not a width this grammar names", 'pt2e', weight_bits=2)
_refused("`symmetric` must be True, False or None", 'pt2e', exc=TypeError, symmetric='yes')
_refused("`qscheme` must be one of ", 'pt2e', exc=TypeError, qscheme=8)
_refused("`group_size` must be a positive int", 'torchao', weight_bits=4, exc=TypeError, group_size='128')
_refused("is not a size", 'torchao', weight_bits=4, group_size=0)
_refused("names no layer", 'x86', weight_bits={})
_refused("`weight_bits keys` must be layer names", 'x86', exc=TypeError, weight_bits={1: 8})
_refused("a per-layer width is 8", 'x86', weight_bits={'fc': 4})
_refused("Unknown backend 'x87'", 'x87')
_refused("`backend` must be a str", exc=TypeError, backend=8)

# --- an unportable cell is allowed, but it warns ---
import warnings
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    test_eq(_resolve_spec('pt2e', symmetric=False).symmetric, False)
assert any('zero-points' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

In [ ]:
# --- the matrix says what torch actually does (this is what keeps it from drifting) ---
import torch
from torch.ao.quantization import get_default_qconfig

_TORCH_AXIS = {torch.per_channel_symmetric: 'per_channel', torch.per_channel_affine: 'per_channel',
               torch.per_tensor_symmetric: 'per_tensor', torch.per_tensor_affine: 'per_tensor'}

for _backend in ('x86', 'fbgemm', 'onednn', 'qnnpack'):
    _qconfig = get_default_qconfig(_backend)
    _cell = PRECISION_SUPPORT[(_backend, 8, 8)]
    # the default weight axis of the matrix is the one the default observer uses
    test_eq(_TORCH_AXIS[_qconfig.weight().qscheme], _cell.default_qscheme)
    # ...and the activations are affine, which is why these cells cannot claim symmetric=True
    test_eq(_qconfig.activation().qscheme, torch.per_tensor_affine)
    test_eq(_cell.symmetries, (False,))
    assert not _cell.exports